# EmotionCLIP-ReID RAF-DB Runbook

Notebook này dùng RAF-DB Basic, tạo landmark offline một lần và lưu mỗi lần train vào một `RUN_ID` có ngày giờ. Official train được tách validation xác định; official test luôn sealed test và không tham gia chọn checkpoint.


## 0. Clone hoặc cập nhật đúng branch `old_branch`

Máy mới dùng `git clone -b old_branch --single-branch`; repo có sẵn sẽ được fetch, checkout và fast-forward đúng branch trước khi chạy.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

JUPYTER_WORKSPACE = Path('/home/jupyter-hault')
REPO_DIR = JUPYTER_WORKSPACE / 'EmotionCLIP-ReID'
GIT_REPO_URL = 'https://github.com/haulth/EmotionCLIP-ReID.git'
REPO_BRANCH = 'old_branch'

JUPYTER_WORKSPACE.mkdir(parents=True, exist_ok=True)
if (REPO_DIR / '.git').exists():
    commands = [
        ['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH],
        ['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH],
        ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH],
    ]
else:
    commands = [[
        'git', 'clone', '-b', REPO_BRANCH, '--single-branch', GIT_REPO_URL, str(REPO_DIR)
    ]]
for command in commands:
    print('Running:', ' '.join(command))
    completed = subprocess.run(command, text=True, capture_output=True, check=True)
    if completed.stdout.strip():
        print(completed.stdout.strip())

REPO = REPO_DIR.resolve()
os.chdir(REPO)
branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
assert branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, got {branch}'
print('Repo:', REPO)
print('Branch:', branch)
print('Python:', sys.executable)


## 1. Kiểm tra package tối thiểu

In [ ]:
import importlib.util
import subprocess
import sys

required = ["torch", "PIL", "yaml", "numpy","kaggle"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
print("Missing:", missing)
if missing:
    print("Cài environment theo environment_emotionclip_cuda.yml hoặc cài các package còn thiếu trước khi train.")

# Kaggle chỉ cần nếu bạn muốn tải bằng Kaggle thay vì upload archive RAF-DB sẵn.
print("Kaggle module:", importlib.util.find_spec("kaggle") is not None)

## 2. Process RAF-DB đã có sẵn

Notebook dùng trực tiếp `data/RAF-DB`. Converter tạo validation xác định từ official train và giữ official test làm test sealed; không remap official test thành validation.


In [ ]:
from pathlib import Path
import subprocess
import sys
REPO = Path.cwd()
RAF_ROOT = REPO / "data" / "RAF-DB"
RAF_MANIFEST = RAF_ROOT / "manifest.jsonl"

if not RAF_ROOT.exists():
    raise FileNotFoundError(f"Không thấy RAF-DB tại {RAF_ROOT}")
for required in ["train_labels.csv", "test_labels.csv"]:
    if not (RAF_ROOT / required).exists():
        raise FileNotFoundError(f"Thiếu {RAF_ROOT / required}")

convert_cmd = [
    sys.executable,
    "tools/convert_rafdb_to_emotion_jsonl.py",
    "--raf-root",
    str(RAF_ROOT),
    "--output",
    str(RAF_MANIFEST),
    "--root-dir",
    str(RAF_ROOT),
]
print("Converting existing data/RAF-DB -> EmotionCLIP manifest")
subprocess.run(convert_cmd, cwd=REPO, check=True)
print("RAF-DB manifest ready:", RAF_MANIFEST)


## 3. Kiểm tra manifest và phân bố split

In [ ]:
import json
from collections import Counter

assert RAF_MANIFEST.exists(), f"Missing manifest: {RAF_MANIFEST}"
records = [json.loads(line) for line in RAF_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Total records:", len(records))
print("Split counts:", Counter(record["split"] for record in records))
print("Official split counts:", Counter(record.get("official_split") for record in records))
print("Emotion counts:", Counter(record["emotion"] for record in records))
print("Split protocol:", records[0].get("split_protocol"))
records[:3]

## 3.1. Khóa ngữ cảnh run theo ngày giờ

Chạy cell đúng một lần cho mỗi run. Output cuối chưa được tạo ở đây; visual trước train được giữ trong staging cùng `RUN_ID` để không phá kiểm tra immutable của training.


In [ ]:
from datetime import datetime
from pathlib import Path
import yaml

from utils.notebook_run import prepare_notebook_staging, timestamped_run_id

RAF_CONFIG = REPO / 'configs/emotion/vit_b16_emotionclip_rafdb_quick.yml'
with RAF_CONFIG.open('r', encoding='utf-8') as handle:
    TRAIN_CFG = yaml.safe_load(handle) or {}
SEED = int(TRAIN_CFG['SOLVER'].get('SEED', 1234))
STAGE1_EPOCHS = 200
STAGE2_EPOCHS = 200
RELIABILITY_WARMUP_EPOCHS = min(10, STAGE2_EPOCHS)
assert STAGE1_EPOCHS >= 1 and STAGE2_EPOCHS >= 1
assert 1 <= RELIABILITY_WARMUP_EPOCHS <= STAGE2_EPOCHS

BASE_MANIFEST = RAF_MANIFEST.resolve()
ACTIVE_MANIFEST = BASE_MANIFEST.with_name('manifest_anatomy.jsonl')
if not ACTIVE_MANIFEST.is_file():
    raise FileNotFoundError(f'Chưa có {ACTIVE_MANIFEST}; hãy chạy notebook build_landmarks trước.')
OUTPUT_ROOT = REPO / 'outputs' / f'emotionclip_rafdb_s1-{STAGE1_EPOCHS}_s2-{STAGE2_EPOCHS}'
RUN_ID_OVERRIDE = ''
RUN_STARTED_AT = datetime.now().astimezone()
RUN_ID = RUN_ID_OVERRIDE.strip() or timestamped_run_id('rafdb', seed=SEED, now=RUN_STARTED_AT)
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
NOTEBOOK_STAGING_DIR = prepare_notebook_staging(REPO, RUN_ID)
NOTEBOOK_VISUAL_STAGE_DIR = NOTEBOOK_STAGING_DIR / 'visuals'

print('RUN_STARTED_AT:', RUN_STARTED_AT.isoformat())
print('RUN_ID:', RUN_ID)
print('OUTPUT_DIR (training sẽ tạo):', OUTPUT_DIR)
print('Notebook staging:', NOTEBOOK_STAGING_DIR)


### 3.2. Visual phân bố dữ liệu

Cell vẽ train/val/test và lưu ảnh vào staging mang đúng `RUN_ID`; sau train ảnh được publish vào `OUTPUT_DIR/visuals`.


In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

CANONICAL_ORDER = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']
emotion_order = [emotion for emotion in CANONICAL_ORDER if any(record['emotion'] == emotion for record in records)]
split_order = [split for split in ['train', 'val', 'test'] if any(record['split'] == split for record in records)]
split_emotion_counts = {
    split: Counter(record['emotion'] for record in records if record['split'] == split)
    for split in split_order
}
total_emotion_counts = Counter(record['emotion'] for record in records)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
x = np.arange(len(emotion_order))
bottom = np.zeros(len(emotion_order), dtype=float)
colors = {'train': '#4c78a8', 'val': '#f58518', 'test': '#54a24b'}
for split in split_order:
    values = np.array([split_emotion_counts[split][emotion] for emotion in emotion_order], dtype=float)
    axes[0].bar(x, values, bottom=bottom, label=split, color=colors[split])
    bottom += values
axes[0].set_xticks(x, emotion_order, rotation=30, ha='right')
axes[0].set_ylabel('Images')
axes[0].set_title('Label distribution by split')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25)

total_values = [total_emotion_counts[emotion] for emotion in emotion_order]
y = np.arange(len(emotion_order))
axes[1].barh(y, total_values, color='#6a9f58')
axes[1].set_yticks(y, emotion_order)
axes[1].invert_yaxis()
axes[1].set_xlabel('Images')
axes[1].set_title('Total images per emotion')
axes[1].grid(axis='x', alpha=0.25)
plt.tight_layout()
distribution_path = NOTEBOOK_VISUAL_STAGE_DIR / 'rafdb_train_distribution.png'
fig.savefig(distribution_path, dpi=180, bbox_inches='tight')
print('Staged:', distribution_path)
plt.show()


## 3.3. Preview ảnh mẫu


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_records = [record for record in records if record['split'] == 'train'][:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.ravel():
    ax.axis('off')
for ax, record in zip(axes.ravel(), sample_records):
    image_path = RAF_ROOT / record['image_path']
    ax.imshow(Image.open(image_path).convert('RGB'))
    ax.set_title(f"{record['split']} / {record['emotion']}")
plt.tight_layout()
preview_path = NOTEBOOK_VISUAL_STAGE_DIR / 'rafdb_train_preview.png'
fig.savefig(preview_path, dpi=180, bbox_inches='tight')
print('Staged:', preview_path)
plt.show()


### 4.1 Gallery mẫu cân bằng theo lớp

Cell này lấy một ảnh đại diện cho mỗi emotion để kiểm tra nhanh đường dẫn ảnh và nhãn sau khi convert manifest.

In [ ]:
import random
from collections import defaultdict
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

rng = random.Random(1234)
records_by_emotion = defaultdict(list)
for record in records:
    if record["split"] == "train":
        records_by_emotion[record["emotion"]].append(record)

sampled_records = []
for emotion in emotion_order:
    candidates = records_by_emotion.get(emotion, [])
    if candidates:
        sampled_records.append(rng.choice(candidates))

cols = min(4, max(1, len(sampled_records)))
rows = int(np.ceil(len(sampled_records) / cols)) if sampled_records else 1
fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.6 * rows))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")

for ax, record in zip(axes, sampled_records):
    image_path = RAF_ROOT / record["image_path"]
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(f"{record['emotion']}\n{Path(record['image_path']).name}", fontsize=10)
    ax.axis("off")
fig.suptitle(f'RAF-DB balanced train gallery | {RUN_ID}', y=1.01)
plt.tight_layout()
gallery_path = NOTEBOOK_VISUAL_STAGE_DIR / 'rafdb_train_balanced_gallery.png'
fig.savefig(gallery_path, dpi=180, bbox_inches='tight')
print('Staged:', gallery_path)
plt.show()


## 5. Train RAF-DB

Chỉnh `STAGE1_EPOCHS`, `STAGE2_EPOCHS` và `RELIABILITY_WARMUP_EPOCHS` ngay trong cell bên dưới; các giá trị sẽ override YAML khi chạy training.

In [ ]:
import os
import subprocess
import sys
from datetime import datetime

from utils.notebook_progress import stream_process_output
from utils.notebook_run import publish_notebook_artifacts

GPU_ID = 1
train_cmd = [
    sys.executable, '-u', 'train_emotionclip.py',
    '--gpu', str(GPU_ID),
    '--run-id', RUN_ID,
    '--config_file', str(RAF_CONFIG),
    'SOLVER.STAGE1.MAX_EPOCHS', str(STAGE1_EPOCHS),
    'SOLVER.STAGE2.MAX_EPOCHS', str(STAGE2_EPOCHS),
    'SOLVER.STAGE2.RELIABILITY_WARMUP_EPOCHS', str(RELIABILITY_WARMUP_EPOCHS),
    'DATASETS.MANIFEST', str(ACTIVE_MANIFEST),
    'DATASETS.ROOT_DIR', str(RAF_ROOT),
    'OUTPUT_DIR', str(OUTPUT_ROOT),
    'TRAIN.PROGRESS_BAR', 'true',
]
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['EMOTIONCLIP_PROGRESS'] = '1'
print('RUN_ID:', RUN_ID)
print('Running:', ' '.join(train_cmd), flush=True)
process = subprocess.Popen(
    train_cmd, cwd=REPO, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
log_chunks = stream_process_output(process)
return_code = process.wait()
RUN_FINISHED_AT = datetime.now().astimezone()

if OUTPUT_DIR.is_dir() and (OUTPUT_DIR / 'provenance.json').is_file():
    notebook_payload = publish_notebook_artifacts(
        OUTPUT_DIR,
        NOTEBOOK_STAGING_DIR,
        console_text=''.join(log_chunks),
        metadata={
            'run_id': RUN_ID,
            'started_at': RUN_STARTED_AT.isoformat(),
            'finished_at': RUN_FINISHED_AT.isoformat(),
            'return_code': return_code,
            'command': train_cmd,
            'manifest': str(ACTIVE_MANIFEST),
        },
    )
    print('Notebook artifacts:', notebook_payload['output_dir'])
else:
    failed_log = NOTEBOOK_STAGING_DIR / 'notebook_console_failed.log'
    failed_log.write_text(''.join(log_chunks), encoding='utf-8')
    print('Run chưa khởi tạo OUTPUT_DIR; console được giữ tại:', failed_log)

if return_code != 0:
    print('\\n'.join(''.join(log_chunks).splitlines()[-80:]))
    raise subprocess.CalledProcessError(return_code, train_cmd)
print('Training finished:', OUTPUT_DIR)


## 6. Xem metrics tốt nhất hoặc epoch mới nhất

In [ ]:
from utils.notebook_metrics import load_validation_metrics, print_validation_summary

RUN_VISUAL_DIR = OUTPUT_DIR / 'visuals'
RUN_VISUAL_DIR.mkdir(parents=True, exist_ok=True)
validation_metrics = load_validation_metrics(candidate_output_dirs=[OUTPUT_DIR])
RESULT_DIR = validation_metrics['result_dir']
metric_files = validation_metrics['metric_files']
metric_history = validation_metrics['metric_history']
metrics_by_epoch = validation_metrics['metrics_by_epoch']
metrics_latest = validation_metrics['metrics_latest']
metrics_best = validation_metrics['metrics_best']
best_metric = validation_metrics['best_metric']
print('RUN_ID:', RUN_ID)
print_validation_summary(validation_metrics)


### 6.1 Biểu đồ metric validation theo epoch

Cell này gom `metrics_epoch_*.json` thành training dynamics của validation/test split.

In [ ]:
from utils.notebook_metrics import plot_validation_metric_curves

validation_visuals = dict(validation_metrics)
validation_visuals['result_dir'] = RUN_VISUAL_DIR
plot_validation_metric_curves(validation_visuals, dataset_name='RAF-DB', file_prefix='rafdb')


### 6.2 Biểu đồ loss/accuracy training từ log

Cell này chỉ parse log/CSV nằm trong `OUTPUT_DIR` của `RUN_ID` hiện tại; không đọc report hoặc thư mục output dùng chung.

In [ ]:
from utils.notebook_metrics import load_training_history, plot_training_metric_curves

training_history, training_source = load_training_history(
    [OUTPUT_DIR], csv_candidates=[OUTPUT_DIR / 'training_epoch_losses.csv']
)
plot_training_metric_curves(
    training_history, training_source, RUN_VISUAL_DIR, file_prefix='rafdb'
)


### 6.3 Confusion matrix và per-class F1

Cell này dùng epoch mới nhất đang được chọn trong `metrics_latest`. Chạy lại cell metrics phía trên nếu muốn đổi thư mục/epoch.

In [ ]:
from utils.notebook_metrics import plot_confusion_matrix_and_f1

plot_confusion_matrix_and_f1(
    validation_visuals, dataset_name='RAF-DB', file_prefix='rafdb'
)


## 7. Infer một ảnh validation

In [ ]:
import json
import subprocess
import sys

from PIL import Image
import matplotlib.pyplot as plt

result_dir = OUTPUT_DIR
eval_records = [record for record in records if record['split'] == 'val']
if not eval_records:
    raise RuntimeError('RAF-DB manifest không có validation split')
sample = eval_records[0]
weight = OUTPUT_DIR / 'best_emotionclip.pth'
image_path = RAF_ROOT / sample['image_path']
assert weight.is_file(), f'Không thấy checkpoint trong run {RUN_ID}: {weight}'

print('Ground truth:', sample['emotion'])
print('Image:', image_path)
print('Weight:', weight)
completed = subprocess.run(
    [
        sys.executable, 'infer_emotionclip.py',
        '--config_file', str(RAF_CONFIG),
        '--weight', str(weight),
        '--image', str(image_path),
    ],
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
)
print(completed.stdout)
json_start = completed.stdout.find('{')
result = json.loads(completed.stdout[json_start:]) if json_start >= 0 else {}
probabilities = result.get('probabilities', {})
names = list(probabilities.keys())
values = [probabilities[name] for name in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [1.0, 1.35]})
axes[0].imshow(Image.open(image_path).convert('RGB'))
axes[0].set_title(
    f"GT: {sample['emotion']}\nPred: {result.get('emotion')} | U={result.get('uncertainty', 0):.3f}"
)
axes[0].axis('off')
y = range(len(names))
colors = ['#2e7d32' if name == result.get('emotion') else '#607d8b' for name in names]
axes[1].barh(y, values, color=colors)
axes[1].set_yticks(list(y), names)
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Probability')
axes[1].set_title('Inference probabilities')
axes[1].invert_yaxis()
plt.tight_layout()
infer_path = RUN_VISUAL_DIR / 'rafdb_inference_example.png'
fig.savefig(infer_path, dpi=180, bbox_inches='tight')
print('Saved:', infer_path)
plt.show()
